# Part 4 — Gemini note classification and recommendation

Use a free Google AI Studio key. **Do not commit the key.** Use Colab Secrets or runtime input; never save API key in notebook output. Brief asks first 10 notes, strict allowed labels, Pydantic validation, intentional bad replies, and recommendations merged to appointments.

In [ ]:
# In Colab: !pip install -q google-genai pydantic
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError
import pandas as pd, json, time
API_KEY=input('Paste Gemini API key (not saved in file): ').strip()
client=genai.Client(api_key=API_KEY)
MODEL='gemini-2.0-flash' # if retired, choose current Flash model in AI Studio
allowed=['forgot_or_no_reminder','distance_or_transport','cost_or_fee','illness_or_emergency','work_or_schedule','other']
class FollowupInsight(BaseModel):
    root_cause:str=Field(description='One allowed root cause label')
    confidence:float=Field(ge=0,le=1)
    summary:str=Field(min_length=5,max_length=250)
    recommended_action:str=Field(min_length=5)
SYSTEM='Classify patient no-show note without guessing. Return root_cause only from: '+', '.join(allowed)+'. If unclear choose other. Give a concise summary and practical respectful action.'
def analyse_note(note_text):
    prompt='Note: '+note_text+'\nReturn JSON with root_cause, confidence, summary, recommended_action.'
    resp=client.models.generate_content(model=MODEL,contents=prompt,config=types.GenerateContentConfig(system_instruction=SYSTEM,temperature=0,response_mime_type='application/json',response_schema=FollowupInsight))
    return json.loads(resp.text)
print('Function ready. Call analyse_note(text) to classify one note.')

Task 4.2 — Validate each note, skip blank notes without an API call, and retain raw replies for invalid rows. Rate-limit calls to respect the free tier. Use the first 10 notes only.

In [ ]:
notes=pd.read_csv('data/followup_notes.csv').head(10)
results=[]
for _,r in notes.iterrows():
    text_note=str(r.note_text).strip()
    if not text_note or text_note.lower()=='nan':
        results.append({'note_id':r.note_id,'appointment_id':r.appointment_id,'root_cause':'other','confidence':0.0,'summary':'No note recorded','recommended_action':'No follow-up action; missing note.','validation_status':'skipped_blank','raw_response':''}); continue
    try:
        raw_reply=analyse_note(text_note)
        parsed=FollowupInsight.model_validate(raw_reply)
        if parsed.root_cause not in allowed: raise ValueError('root_cause outside allow-list')
        results.append({'note_id':r.note_id,'appointment_id':r.appointment_id,**parsed.model_dump(),'validation_status':'valid','raw_response':json.dumps(raw_reply)})
    except Exception as e:
        results.append({'note_id':r.note_id,'appointment_id':r.appointment_id,'root_cause':'other','confidence':0.0,'summary':'Invalid model response retained for review','recommended_action':'Review the raw response and follow up manually.','validation_status':'invalid','raw_response':str(e)})
    time.sleep(12) # conservative pacing; may be adjusted to free-tier limits
notes_analyzed=pd.DataFrame(results)
notes_analyzed.to_csv('notes_analysed.csv',index=False)
notes_analyzed.head()

Task 4.3 — Intentional validation failures and prompt comparison. Demonstrate invalid confidence, missing recommended_action, and prompt variants; count API calls. Do not let validation failures crash the loop.

In [ ]:
try: FollowupInsight.model_validate({'root_cause':'other','confidence':1.7,'summary':'Too high confidence'})
except ValidationError as err: print('Confidence validation caught:',err)
try: FollowupInsight.model_validate({'root_cause':'other','confidence':.5,'summary':'Missing action'})
except ValidationError as err: print('Missing field caught:',err)
print('Allowed labels:',allowed)

Task 4.4 — Merge and report. Join note analysis with appointment records using appointment_id; cross-tab root cause by reminder_sent. Use cautious, association-only language.

In [ ]:
appt=pd.read_csv('appointments_clean.csv')
if 'notes_analyzed' not in globals(): notes_analyzed=pd.read_csv('notes_analysed.csv')
merged=notes_analyzed.merge(appt,on='appointment_id',how='left')
print('Rows merged:',len(merged))
print(pd.crosstab(merged.root_cause,merged.reminder_sent,dropna=False))
print('Counts per reason:\n',merged.root_cause.value_counts())
print('Reminder cross-tab (counts):\n',pd.crosstab(merged.root_cause,merged.reminder_sent))